# 5. Trial windows for cRQA (pairs of participants)

**Run once per session.** Input: the envelopes of `04_spatial_filter` (one per participant and area, 25 Hz). Output: for every **pair × area × trial**, the two envelopes cut to the window both were playing, plus the pair's **role** in that trial. The cRQA itself is done in R with the `crqa` package (`06_crqa.R`).

**Smoothing (optional):** with `CRQA_SMOOTH_HZ` in `config.py` (or `RT_SMOOTH_HZ=1` when running), each envelope is low-passed at that frequency before the windows are cut, and the output goes to `<session>_smooth<X>hz/`. Then run `06_crqa.R <session>_smooth<X>hz`.

This step stays in Python because the windows are cut in **session time** with `rt_bids` (each headset's clock drift and skipped samples are corrected there).

**Design A** (`CLAUDE.md`): the same pair is compared across the roles it had: `teammates` / `opponents` (Team trials), `solo` (Solo trials), `baseline` (motor baseline).

**Steps:** 1. envelopes and pairs → 2. trials and roles → 3. cut the windows in session time → 4. save for R.

In [ ]:
import itertools
import os

import mne
import numpy as np
import pandas as pd

import config as cfg  # also puts rt_bids on the path
import rt_bids
mne.set_log_level("error")

## 0. Session

In [ ]:
# execute_notebooks.py passes the session in RT_SUBJECT; when running by hand, change the default here
SESSION = os.environ.get("RT_SUBJECT", "session03")
# smoothing of the envelopes (config.py: CRQA_SMOOTH_HZ; RT_SMOOTH_HZ overrides it, "0" = none)
SMOOTH_HZ = float(os.environ["RT_SMOOTH_HZ"]) if "RT_SMOOTH_HZ" in os.environ else cfg.CRQA_SMOOTH_HZ
SMOOTH_HZ = SMOOTH_HZ or None
# smoothed windows go to their own folder, e.g. session03_smooth1hz, so both versions can be compared
OUT = cfg.CRQA_ROOT / (SESSION if SMOOTH_HZ is None else f"{SESSION}_smooth{SMOOTH_HZ:g}hz")
OUT.mkdir(parents=True, exist_ok=True)
FS = cfg.ENVELOPE_SFREQ
print(SESSION, "smoothing:", f"{SMOOTH_HZ:g} Hz" if SMOOTH_HZ else "none", "->", OUT)

## 1. Envelopes and pairs

For every participant of the session with a choice in `SPATIAL_FILTER_CHOICE`, the envelopes saved by `04` are loaded. What `04` actually used is read from `<sub>_spatial_filter.csv` and compared with `config.py`: if they differ, `04` has to be rerun first.

A pair is two participants with the same area (e.g. both have a `visual` envelope). Every pair is used once (A = the lower participant number).

In [ ]:
subjects = [s for s in cfg.session_subjects(SESSION) if s in cfg.SPATIAL_FILTER_CHOICE]
envelopes, rows = {}, []
first_samp, clean_sfreq = {}, {}
for sub in subjects:
    used = pd.read_csv(cfg.out_dir(sub) / f"{sub}_spatial_filter.csv")
    for r in used.itertuples():
        wanted = cfg.SPATIAL_FILTER_CHOICE[sub].get(r.area)
        assert wanted == (r.ica, r.component, r.band), (
            f"{sub} {r.area}: config.py says {wanted}, 04 used {(r.ica, r.component, r.band)}: rerun 04")
        raw = mne.io.read_raw(cfg.out_dir(sub) / f"{sub}_{r.area}_{r.band}_{r.ica}_envelope_raw.fif", preload=True)
        assert raw.info["sfreq"] == FS, f"{sub} {r.area}: envelope at {raw.info['sfreq']} Hz, expected {FS}"
        if SMOOTH_HZ:
            # low-pass of the whole envelope (before cutting, so no edge effects in the windows); narrow slope (half
            # of SMOOTH_HZ), as in 04 section 3b: at 1 Hz the -6 dB point is 1.25 Hz. The envelope is a 'misc' channel
            raw.filter(None, SMOOTH_HZ, picks="all", h_trans_bandwidth=SMOOTH_HZ / 2)
        envelopes[(sub, r.area)] = {"raw": raw, "x": raw.get_data()[0], "band": r.band, "ica": r.ica,
                                    "component": int(r.component)}
        rows.append({"subject": sub, "area": r.area, "ica": r.ica, "component": r.component, "band": r.band})
    # the clean data's first sample (500 Hz): the envelope starts there (01 cut the recording before it)
    clean = mne.io.read_raw(cfg.out_dir(sub) / f"{sub}_clean_raw.fif", preload=False)
    first_samp[sub], clean_sfreq[sub] = clean.first_samp, clean.info["sfreq"]

areas = sorted({area for (_, area) in envelopes})
pairs = [(a, b, area) for area in areas
         for a, b in itertools.combinations(sorted(s for (s, ar) in envelopes if ar == area), 2)]
print(f"{len(subjects)} participants with envelopes; areas: {areas}; {len(pairs)} pair x area combinations")
pd.DataFrame(rows)

## 2. Trials and roles

The trials come from the behavioural events of the session (`rt_bids.session_events`, `trial_type == "trial"`), in **session time** (the clock all recordings share). Each person has a row per trial with its start, duration and team (`team_id`, a fact about the trial: teams are reassigned between blocks).

For each pair and trial, the **window both were playing** is used: from the later start to the earlier end. The role:
- `joint` trial, same `team_id` → `teammates`; different → `opponents`;
- `individual` → `solo`; `motor_baseline` → `baseline`.

In [ ]:
events = rt_bids.session_events(cfg.BIDS_ROOT, SESSION)
trials = events[(events["trial_type"] == "trial") & events["condition"].isin(cfg.CONDITIONS)]
trials = trials[["participant_id", "condition", "round", "trial", "team_id", "onset", "duration"]].copy()
trials["end"] = trials["onset"] + trials["duration"]


def role(condition, team_a, team_b):
    if condition == "joint":
        return "teammates" if team_a == team_b else "opponents"
    return {"individual": "solo", "motor_baseline": "baseline"}[condition]


windows = []
for a, b in sorted({(a, b) for a, b, _ in pairs}):
    both = trials[trials["participant_id"] == a].merge(
        trials[trials["participant_id"] == b], on=["condition", "round", "trial"], suffixes=("_a", "_b"))
    for t in both.itertuples():
        windows.append({"participant_a": a, "participant_b": b, "condition": t.condition, "round": int(t.round),
                        "trial": int(t.trial), "team_a": t.team_id_a, "team_b": t.team_id_b,
                        "role": role(t.condition, t.team_id_a, t.team_id_b),
                        "start": max(t.onset_a, t.onset_b), "end": min(t.end_a, t.end_b)})
windows = pd.DataFrame(windows)
print(windows.groupby(["participant_a", "participant_b", "role"]).size().unstack(fill_value=0))

## 3. Cutting the windows in session time

Session time → sample in the envelope, per person:
1. `rt_bids.from_session_time(<recording>, t)` gives the (fractional) sample in the **original** recording at 500 Hz. It corrects for each headset's clock drift against the session clock and for skipped samples (`acq_gap`).
2. The envelope starts at the clean data's first sample (`first_samp`; `01` cut the recording 2 s before the first trial) and has 25 samples per second instead of 500: envelope sample = (original sample − `first_samp`) × 25 / 500.

(`rt_bids.crop_session` is not used: it ignores `first_samp`, so it is wrong on our cropped files.)

Both cuts start at the first whole sample inside the window; the longer one is trimmed to the shorter (they can differ by a sample). The share of each window marked bad (`BAD_amplitude` from `01`) is recorded, not removed, so the analysis can decide.

In [ ]:
def envelope_samples(sub, times):
    # fractional envelope sample(s) of session time(s)
    k = rt_bids.from_session_time(cfg.bdf_path(sub), times)          # original recording, 500 Hz
    return (k - first_samp[sub]) * FS / clean_sfreq[sub]


def bad_fraction(raw, start, stop):
    # share of the samples start..stop (envelope samples) covered by annotations starting with BAD_
    ann = raw.annotations
    offset = raw.first_time if ann.orig_time is not None else 0.0
    bad = np.zeros(stop - start, bool)
    for onset, duration, description in zip(ann.onset, ann.duration, ann.description):
        if description.startswith("BAD_"):
            i0 = int(np.floor((onset - offset) * FS)) - start
            i1 = int(np.ceil((onset - offset + duration) * FS)) - start
            bad[max(i0, 0):max(min(i1, len(bad)), 0)] = True
    return float(bad.mean())


table, series = [], []
for w in windows.itertuples():
    ia = envelope_samples(w.participant_a, [w.start, w.end])
    ib = envelope_samples(w.participant_b, [w.start, w.end])
    if np.isnan(ia).any() or np.isnan(ib).any():
        print(f"skipped (outside a recording or in a gap): {w.participant_a}-{w.participant_b} {w.condition} r{w.round} t{w.trial}")
        continue
    sa, sb = int(np.ceil(ia[0])), int(np.ceil(ib[0]))
    n = min(int(np.floor(ia[1])) - sa, int(np.floor(ib[1])) - sb)
    for area in areas:
        if (w.participant_a, area) not in envelopes or (w.participant_b, area) not in envelopes:
            continue
        ea, eb = envelopes[(w.participant_a, area)], envelopes[(w.participant_b, area)]
        assert sa >= 0 and sb >= 0 and sa + n <= len(ea["x"]) and sb + n <= len(eb["x"]), "window outside the envelope"
        window_id = len(table) + 1
        table.append({"window_id": window_id, "session": SESSION, "area": area,
                      "pair": f"{w.participant_a}_{w.participant_b}",
                      **{k: v for k, v in w._asdict().items() if k != "Index"},
                      "ica_a": ea["ica"], "component_a": ea["component"], "ica_b": eb["ica"],
                      "component_b": eb["component"], "band": ea["band"],
                      "n_samples": n, "duration_s": n / FS,
                      "bad_fraction_a": bad_fraction(ea["raw"], sa, sa + n),
                      "bad_fraction_b": bad_fraction(eb["raw"], sb, sb + n)})
        series.append(pd.DataFrame({"window_id": window_id, "sample": np.arange(n),
                                    "x_a": ea["x"][sa:sa + n], "x_b": eb["x"][sb:sb + n]}))

table = pd.DataFrame(table)
print(table.groupby(["area", "role"]).agg(n=("n_samples", "size"), duration_s=("duration_s", "median"),
                                          bad_max_a=("bad_fraction_a", "max"), bad_max_b=("bad_fraction_b", "max")).round(3))

## 4. Save for R

- `windows.csv`: one row per pair × area × trial: `window_id`, the pair and their components, condition, round, trial, role, the window in session time (`start`, `end`), its length and the share of bad time.
- `envelopes.csv`: the two cut envelopes of every window (`window_id`, `sample`, `x_a`, `x_b`; in volts, `06_crqa.R` z-scores them).

Then run `06_crqa.R` for the session.

In [ ]:
table.to_csv(OUT / "windows.csv", index=False)
pd.concat(series).to_csv(OUT / "envelopes.csv", index=False, float_format="%.6e")
print(f"saved {len(table)} windows to", OUT)